# Week 7 Assignment – Data Engineering with Pandas and Delta Lake


## Introduction

For this week's assignment I used the **Superstore sales dataset** (`customer_master.csv`), which is a well-known retail dataset with 9,994 order-line records. Each row represents one product sold as part of an order — it has order info (Order ID, Order Date, Ship Date), customer info (Customer ID, Name, Segment), location info (City, State, Region), product info (Category, Sub-Category, Product Name), and the numbers we care about most: Sales, Quantity, Discount and Profit.

In Part 1 I explored and cleaned this dataset with Pandas. In Part 2 I moved into Delta Lake on Databricks to practice incremental loading using `MERGE`.


## Objective

- Load and explore the raw Superstore CSV dataset using Pandas
- Check for missing values and duplicate rows (and handle them if any are found)
- Perform basic filtering and column selection
- Create a derived column representing the order's total amount
- Save the cleaned dataset as `cleaned_customer.csv`
- Build a per-customer summary table and load it into a Delta Table using PySpark
- Create an incremental dataset with updated existing customers + new customers
- Perform a `MERGE INTO` operation to upsert the incremental data
- Validate the final table (row count, duplicate check)

## Part 1 – Pandas

### Import Libraries

In [0]:
# importing the libraries I need for Part 1
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

### Load Dataset



In [0]:
# loading the CSV file into a dataframe
df = pd.read_csv("/Volumes/workspace/default/week7_data/customer_master.csv",
    encoding="latin1")

df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Explore Dataset


In [0]:
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [0]:
df.tail()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,Florida,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.248,3,0.2,4.1028
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.960,2,0.0,15.6332
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.576,2,0.2,19.3932
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,California,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.600,4,0.0,13.3200
9993,9994,CA-2017-119914,5/4/2017,5/9/2017,Second Class,CC-12220,Chris Cortes,Consumer,United States,Westminster,California,92683,West,OFF-AP-10002684,Office Supplies,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.160,2,0.0,72.9480


In [0]:
df.shape

(9994, 21)

In [0]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='str')

In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   Order Date     9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  Category       9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  Sales          9994 non-null   float64
 18  Quantity

In [0]:
df.describe()

,Row ID,Postal Code,Sales,Quantity,Discount,Profit
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,229.858001,3.789574,0.156203,28.656896
std,2885.163629,32063.693350,623.245101,2.225110,0.206452,234.260108
min,1.000000,1040.000000,0.444000,1.000000,0.000000,-6599.978000
25%,2499.250000,23223.000000,17.280000,2.000000,0.000000,1.728750
50%,4997.500000,56430.500000,54.490000,3.000000,0.200000,8.666500
75%,7495.750000,90008.000000,209.940000,5.000000,0.200000,29.364000
max,9994.000000,99301.000000,22638.480000,14.000000,0.800000,8399.976000


### Handle Missing Values

Checking for nulls across every column. As mentioned in the introduction, this dataset turned out to be already clean — there are no missing values at all. I'm still running the check (and would have handled them below if any showed up) because skipping this step just because you *assume* the data is clean is a bad habit.

If there had been missing values, this is roughly how I would have handled each type of column:
- Text columns (like City, Region) → fill with `"Unknown"`
- Numeric columns like Sales or Quantity → these are critical for calculations, so rows missing them would likely need to be dropped instead of guessed


In [0]:
# checking how many nulls are present in each column
df.isnull().sum()

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

In [0]:
# double-checking with a single overall number
print("Total missing values in the dataset:", df.isnull().sum().sum())

Total missing values in the dataset: 0


### Remove Duplicate Records

Checking for fully duplicated rows. Again, none were found in this dataset, but I'm running `drop_duplicates()` anyway as a safety step — this is standard practice even on data you believe is already clean, since it costs nothing to run and protects against silent mistakes.



In [0]:
print("Duplicate rows before cleaning:", df.duplicated().sum())

df = df.drop_duplicates()

print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Total rows remaining:", df.shape[0])

Duplicate rows before cleaning: 0
Duplicate rows after cleaning: 0
Total rows remaining: 9994


### Filter Data and Select Columns

Just to demonstrate basic filtering and column selection, I'm picking a few important columns and filtering for high-value orders (Sales above 1000).

In [0]:
# selecting a few important columns
subset_cols = df[['Order ID', 'Customer Name', 'Category', 'Sub-Category', 'Sales', 'Quantity', 'Profit']]
subset_cols.head()

,Order ID,Customer Name,Category,Sub-Category,Sales,Quantity,Profit
0,CA-2016-152156,Claire Gute,Furniture,Bookcases,261.9600,2,41.9136
1,CA-2016-152156,Claire Gute,Furniture,Chairs,731.9400,3,219.5820
2,CA-2016-138688,Darrin Van Huff,Office Supplies,Labels,14.6200,2,6.8714
3,US-2015-108966,Sean O'Donnell,Furniture,Tables,957.5775,5,-383.0310
4,US-2015-108966,Sean O'Donnell,Office Supplies,Storage,22.3680,2,2.5164


In [0]:
# filtering for high-value orders
high_value_orders = df[df['Sales'] > 1000]
print("Number of high-value orders (Sales > 1000):", high_value_orders.shape[0])
high_value_orders.head()

Number of high-value orders (Sales > 1000): 468


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
10,11,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West,FUR-TA-10001539,Furniture,Tables,Chromcraft Rectangular Conference Tables,1706.184,9,0.2,85.3092
24,25,CA-2015-106320,9/25/2015,9/30/2015,Standard Class,EB-13870,Emily Burns,Consumer,United States,Orem,Utah,84057,West,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,1044.630,3,0.0,240.2649
27,28,US-2015-150630,9/17/2015,9/21/2015,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",3083.430,7,0.5,-1665.0522
35,36,CA-2016-117590,12/8/2016,12/10/2016,First Class,GH-14485,Gene Hale,Corporate,United States,Richardson,Texas,75080,Central,TEC-PH-10004977,Technology,Phones,GE 30524EE4,1097.544,7,0.2,123.4737
54,55,CA-2016-105816,12/11/2016,12/17/2016,Standard Class,JM-15265,Janet Molinari,Corporate,United States,New York City,New York,10024,East,TEC-PH-10002447,Technology,Phones,AT&T CL83451 4-Handset Telephone,1029.950,5,0.0,298.6855


### Create Derived Column – Total Amount

The assignment asks for a derived column `total_amount = price * quantity`. This dataset doesn't have a separate "price per unit" column — instead it has `Sales`, which is already the total sales value for that order line (price × quantity, after discount).

To follow the assignment's requirement properly, I first work backwards to get a **Unit Price** (`Sales / Quantity`), and then recreate **Total Amount** as `Unit Price * Quantity`. This should land back on the original `Sales` value (small rounding differences aside), which is a nice way to double check the logic is correct.

In [0]:
# deriving unit price from Sales and Quantity
df['Unit Price'] = (df['Sales'] / df['Quantity']).round(2)

# recreating total_amount the way the assignment asks: price * quantity
df['Total Amount'] = (df['Unit Price'] * df['Quantity']).round(2)

# sanity check - Total Amount should be very close to the original Sales column
df['check_diff'] = (df['Total Amount'] - df['Sales']).abs()
print("Max difference between Total Amount and original Sales:", df['check_diff'].max())

df[['Sales', 'Quantity', 'Unit Price', 'Total Amount']].head()

Max difference between Total Amount and original Sales: 0.05600000000004002


,Sales,Quantity,Unit Price,Total Amount
0,261.9600,2,130.98,261.96
1,731.9400,3,243.98,731.94
2,14.6200,2,7.31,14.62
3,957.5775,5,191.52,957.60
4,22.3680,2,11.18,22.36


In [0]:
# dropping the helper column, we don't need it in the final output
df = df.drop(columns=['check_diff'])

### Save Cleaned Dataset

Saving the cleaned dataframe as `cleaned_customer.csv`. This still has one row per order line (9,994 rows) — I'll build a per-customer summary from this in Part 2, since the MERGE exercise works better at the customer level.

In [0]:
df.to_csv(
    "/Volumes/workspace/default/week7_data/cleaned_customer.csv",
    index=False
)

print("Cleaned file saved successfully.")
print("Final Shape:", df.shape)

Cleaned file saved successfully.
Final Shape: (9994, 23)


### Summary of Part 1

- Loaded the raw Superstore dataset (9,994 rows, 21 columns) and explored it using `head()`, `tail()`, `shape`, `columns`, `info()` and `describe()`.
- Checked for missing values and duplicate rows — found none, but ran the checks anyway as good practice.
- Selected a subset of columns and filtered for high-value orders as a filtering demo.
- Derived `Unit Price` and `Total Amount` columns (satisfying the `price × quantity` requirement) and verified they reproduce the original `Sales` values.
- Saved the result as `cleaned_customer.csv`.

## Part 2 – Delta Lake (Incremental Processing)

Now moving into the Databricks part of the assignment.

**Important adaptation:** the cleaned dataset from Part 1 is at the *order-line* level (9,994 rows — one row per product per order). The assignment's incremental/MERGE exercise, however, is about *customers* (existing customers being updated + new customers being added). Merging at the order-line level wouldn't really demonstrate that concept properly, since every order line already has a unique Row ID and there's nothing meaningful to "update."

So before loading into Delta, I first build a **customer summary table** — one row per customer, aggregating their total orders, quantity, sales and profit. This is the table I load into Delta Lake and perform the MERGE against, which matches the spirit of the original assignment (existing customers being updated, new customers being inserted).

**Note:** This part is written to run inside a **Databricks notebook**, since Delta Lake needs a Databricks/Spark environment. The CSV files are assumed to be uploaded to a **Databricks Volume**.

### Create Spark Session

On Databricks, `spark` is already available automatically. I'm still including this code so the notebook also works if run outside Databricks.

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week7_DeltaLake_Assignment") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark

### Build the Customer Summary Table

Reading the cleaned order-line data and aggregating it by customer, so we get one row per customer with their total orders, quantity, sales and profit. I'm also picking each customer's most common `Region` as their "home region," since a customer can have orders shipped to more than one region.


In [0]:
cleaned_csv_path = "/Volumes/workspace/default/week7_data/cleaned_customer.csv"

cleaned_df = spark.read.csv(
    cleaned_csv_path,
    header=True,
    inferSchema=True
)

cleaned_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|Unit Price|Total Amount|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+----------+------------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Read cleaned CSV with proper quote escaping
cleaned_csv_path = "/Volumes/workspace/default/week7_data/cleaned_customer.csv"

cleaned_df = spark.read.csv(
    cleaned_csv_path,
    header=True,
    inferSchema=True,
    quote='"',
    escape='"'
)

# 2. Determine each customer's most frequent home region
region_counts = cleaned_df.groupBy("Customer ID", "Region").count()
window_spec = Window.partitionBy("Customer ID").orderBy(F.col("count").desc())

home_region_df = (
    region_counts
    .withColumn("rank", F.row_number().over(window_spec))
    .filter(F.col("rank") == 1)
    .select("Customer ID", F.col("Region").alias("home_region"))
)

# 3. Aggregate customer summary metrics
customer_summary_df = (
    cleaned_df.groupBy("Customer ID", "Customer Name", "Segment")
    .agg(
        F.countDistinct("Order ID").alias("total_orders"),
        F.sum("Quantity").alias("total_quantity"),
        F.round(F.sum("Sales"), 2).alias("total_sales"),
        F.round(F.sum("Profit"), 2).alias("total_profit")
    )
    .join(home_region_df, on="Customer ID", how="left")
)

customer_summary_df.show(10)
print("Total customers:", customer_summary_df.count())

+-----------+----------------+-----------+------------+--------------+-----------+------------+-----------+
|Customer ID|   Customer Name|    Segment|total_orders|total_quantity|total_sales|total_profit|home_region|
+-----------+----------------+-----------+------------+--------------+-----------+------------+-----------+
|   TD-20995|   Tamara Dahlen|   Consumer|           9|            67|    1434.55|       88.19|    Central|
|   AJ-10795| Anthony Johnson|  Corporate|           7|            88|    4501.39|     1158.71|       West|
|   SC-20440|    Shaun Chance|  Corporate|           7|            44|     1875.0|      379.56|       East|
|   SH-20395|  Shahid Hopkins|   Consumer|          10|            45|    2180.72|     -144.52|       East|
|   KL-16555|   Kelly Lampkin|  Corporate|           8|            84|    5016.49|     -182.78|      South|
|   NB-18655|       Nona Balk|  Corporate|           9|            64|     1972.6|      117.64|    Central|
|   AA-10375|    Allen Armol

### Load Customer Summary into a Delta Table

Writing the aggregated customer summary out as a Delta Table.


In [0]:
# Create the schema inside the 'workspace' catalog if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")

# Rename columns to remove spaces (Delta Lake requirement)
customer_summary_df = customer_summary_df.withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Customer Name", "customer_name")

# Define Delta table name inside workspace catalog and default schema
delta_table_name = "workspace.default.customer_summary"

(
    customer_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(delta_table_name)
)

print(f"Successfully created Delta Table: {delta_table_name}")

# Display Delta Table contents to verify
display(spark.table(delta_table_name))

Successfully created Delta Table: workspace.default.customer_summary


customer_id,customer_name,Segment,total_orders,total_quantity,total_sales,total_profit,home_region
TD-20995,Tamara Dahlen,Consumer,9,67,1434.55,88.19,Central
AJ-10795,Anthony Johnson,Corporate,7,88,4501.39,1158.71,West
SC-20440,Shaun Chance,Corporate,7,44,1875.0,379.56,East
SH-20395,Shahid Hopkins,Consumer,10,45,2180.72,-144.52,East
KL-16555,Kelly Lampkin,Corporate,8,84,5016.49,-182.78,South
NB-18655,Nona Balk,Corporate,9,64,1972.6,117.64,Central
AA-10375,Allen Armold,Consumer,9,41,1056.39,277.38,East
BM-11650,Brian Moss,Corporate,11,95,7294.19,2199.28,Central
DP-13390,Dennis Pardue,Home Office,9,50,5480.72,1571.83,West
TB-21400,Tom Boeckenhauer,Consumer,7,55,9133.99,2798.37,West


In [0]:
# reading back the delta table to confirm it was written correctly
customer_delta_df = spark.table("workspace.default.customer_summary")
customer_delta_df.show(5)
print("Row count in Delta table:", customer_delta_df.count())

+-----------+---------------+---------+------------+--------------+-----------+------------+-----------+
|customer_id|  customer_name|  Segment|total_orders|total_quantity|total_sales|total_profit|home_region|
+-----------+---------------+---------+------------+--------------+-----------+------------+-----------+
|   TD-20995|  Tamara Dahlen| Consumer|           9|            67|    1434.55|       88.19|    Central|
|   AJ-10795|Anthony Johnson|Corporate|           7|            88|    4501.39|     1158.71|       West|
|   SC-20440|   Shaun Chance|Corporate|           7|            44|     1875.0|      379.56|       East|
|   SH-20395| Shahid Hopkins| Consumer|          10|            45|    2180.72|     -144.52|       East|
|   KL-16555|  Kelly Lampkin|Corporate|           8|            84|    5016.49|     -182.78|      South|
+-----------+---------------+---------+------------+--------------+-----------+------------+-----------+
only showing top 5 rows
Row count in Delta table: 793


### Create the Incremental Dataset

For the MERGE demo, `customer_incremental.csv` has two kinds of records:

- **8 existing customers** whose totals have changed (simulating them placing a new order — their `total_orders`, `total_quantity`, `total_sales` and `total_profit` all go up)
- **5 new customers** (Customer IDs starting with `ZZ-9...`) who don't exist in the customer summary table yet

This mimics a real scenario where a new batch of order activity comes in and needs to be merged into the customer table, instead of recalculating everything from scratch.



In [0]:
incremental_csv_path = "/Volumes/workspace/default/week7_data/customer_incremental.csv"

incremental_df = spark.read.csv(incremental_csv_path, header=True, inferSchema=True)
incremental_df.show(truncate=False)

+-----------+---------------+-----------+-----------+------------+--------------+-----------+------------+
|Customer ID|Customer Name  |Segment    |home_region|total_orders|total_quantity|total_sales|total_profit|
+-----------+---------------+-----------+-----------+------------+--------------+-----------+------------+
|BP-11290   |Beth Paige     |Consumer   |Central    |8           |54            |2895.58    |-223.76     |
|MO-17950   |Michael Oakman |Consumer   |Central    |3           |16            |244.28     |-63.61      |
|JM-15265   |Janet Molinari |Corporate  |Central    |6           |57            |3054.87    |812.81      |
|TS-21655   |Trudy Schmidt  |Consumer   |Central    |6           |53            |3423.31    |233.02      |
|RW-19630   |Rob Williams   |Corporate  |Central    |10          |57            |3889.74    |838.83      |
|JW-16075   |Julia West     |Consumer   |South      |5           |40            |1628.62    |199.66      |
|LM-17065   |Liz MacKendrick|Consumer

### MERGE Operation

**Why MERGE is used:**
Without MERGE, we'd have to manually figure out which customer_ids already exist (to update their totals) and which ones are new (to insert), then run two separate operations. MERGE does both in a single, atomic step.

**What happens during UPDATE:**
If a `Customer ID` from the incremental file already exists in the Delta table, MERGE updates that customer's `total_orders`, `total_quantity`, `total_sales` and `total_profit` with the new values from the incremental file.

**What happens during INSERT:**
If a `Customer ID` from the incremental file does NOT exist in the Delta table, MERGE simply inserts it as a brand new customer row.

**Example – before and after MERGE:**

| Stage | total_orders | total_quantity | total_sales |
|---|---|---|---|
| Before MERGE (old record) | 5 | 18 | 2475.08 |
| Incoming record (incremental file) | 6 | 21 | 2895.58 |
| After MERGE (updated record) | 6 | 21 | 2895.58 |

And for a brand-new customer like `ZZ-90001 – Nikita Solanki`, there is no "before" row at all — MERGE simply inserts it as a new record.


In [0]:
from delta.tables import DeltaTable

# Rename columns in incremental_df to match the Delta table schema
incremental_df = incremental_df.withColumnRenamed("Customer ID", "customer_id") \
    .withColumnRenamed("Customer Name", "customer_name")

# Load the Delta table by name (not path)
delta_table = DeltaTable.forName(spark, delta_table_name)

delta_table.alias("target").merge(
    incremental_df.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(set={
    "customer_name": "source.customer_name",
    "Segment": "source.Segment",
    "home_region": "source.home_region",
    "total_orders": "source.total_orders",
    "total_quantity": "source.total_quantity",
    "total_sales": "source.total_sales",
    "total_profit": "source.total_profit"
}).whenNotMatchedInsert(values={
    "customer_id": "source.customer_id",
    "customer_name": "source.customer_name",
    "Segment": "source.Segment",
    "home_region": "source.home_region",
    "total_orders": "source.total_orders",
    "total_quantity": "source.total_quantity",
    "total_sales": "source.total_sales",
    "total_profit": "source.total_profit"
}).execute()

print("MERGE completed successfully.")

MERGE completed successfully.


### Validation


In [0]:
merged_df = spark.table("workspace.default.customer_summary")

print("Total customers after MERGE:", merged_df.count())
print("Expected: 793 original customers + 5 new customers = 798")

duplicate_check = merged_df.groupBy("customer_id").count().filter("count > 1")
print("Number of duplicate Customer IDs after MERGE:", duplicate_check.count())
duplicate_check.show()

Total customers after MERGE: 798
Expected: 793 original customers + 5 new customers = 798
Number of duplicate Customer IDs after MERGE: 0
+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



### Final Output


In [0]:
final_result = spark.table("workspace.default.customer_summary").orderBy("customer_id")
final_result.show(50, truncate=False)

+-----------+--------------------+-----------+------------+--------------+-----------+------------+-----------+
|customer_id|customer_name       |Segment    |total_orders|total_quantity|total_sales|total_profit|home_region|
+-----------+--------------------+-----------+------------+--------------+-----------+------------+-----------+
|AA-10315   |Alex Avila          |Consumer   |5           |30            |5563.56    |-362.88     |Central    |
|AA-10375   |Allen Armold        |Consumer   |9           |41            |1056.39    |277.38      |East       |
|AA-10480   |Andrew Allen        |Consumer   |4           |36            |1790.51    |435.83      |Central    |
|AA-10645   |Anna Andreadi       |Consumer   |6           |64            |5086.93    |857.8       |East       |
|AB-10015   |Aaron Bergman       |Consumer   |3           |13            |886.16     |129.35      |West       |
|AB-10060   |Adam Bellavance     |Home Office|8           |56            |7755.62    |2054.59     |East 

In [0]:
# double-checking the merge worked correctly by looking at just the customers that were touched
touched_ids = [row["customer_id"] for row in incremental_df.select("customer_id").collect()]
final_result.filter(final_result["customer_id"].isin(touched_ids)).show(20, truncate=False)

+-----------+---------------+-----------+------------+--------------+-----------+------------+-----------+
|customer_id|customer_name  |Segment    |total_orders|total_quantity|total_sales|total_profit|home_region|
+-----------+---------------+-----------+------------+--------------+-----------+------------+-----------+
|BP-11290   |Beth Paige     |Consumer   |8           |54            |2895.58    |-223.76     |Central    |
|JM-15265   |Janet Molinari |Corporate  |6           |57            |3054.87    |812.81      |Central    |
|JW-16075   |Julia West     |Consumer   |5           |40            |1628.62    |199.66      |South      |
|LM-17065   |Liz MacKendrick|Consumer   |6           |35            |1686.85    |33.42       |South      |
|MG-17695   |Maureen Gnade  |Consumer   |4           |29            |2322.72    |-187.89     |East       |
|MO-17950   |Michael Oakman |Consumer   |3           |16            |244.28     |-63.61      |Central    |
|RW-19630   |Rob Williams   |Corporat

## Conclusion

This assignment gave good practice with a real-world-style dataset instead of a toy one. A few things stood out:

- The Superstore dataset turned out to already be clean, which was a good reminder that the point of exploration and null/duplicate checks is to *verify* the data, not to assume there's a problem.
- Since the dataset didn't have a direct "price" column, I had to think about how to derive one sensibly (`Sales / Quantity`) rather than just following the assignment's instructions literally.
- The biggest adaptation was realizing that MERGE makes much more sense at the *customer* level than at the raw order-line level, since order lines don't naturally get "updated" the way customer totals do. Building a customer summary table first made the MERGE exercise actually meaningful.

**Key learnings:**
- Real datasets don't always match the shape you expect from an assignment brief — sometimes you need to aggregate or reshape the data to make the exercise (like MERGE) actually demonstrate the right concept.
- `MERGE INTO` = UPDATE (for matching records) + INSERT (for new records), done in a single step.
- Always validate row counts and duplicates after any load, to catch mistakes early.